# 00 · Environment test (Lab 7, Prompt 08)

**Before running:** Settings → Accelerator **GPU T4 x2**; Internet **on**. No secrets or tokens are needed
(the model and the dataset are public and not gated).

**Run cells 1–4 in order, then copy the whole output of every cell back to Claude.** Nothing here touches
the test set: the smoke test uses 5 items of the 25-item debug set.

| Cell | What it does | Expected output |
|---|---|---|
| 1 Diagnostics | Python, GPU, driver/CUDA, disk, preinstalled library versions, model commit hash | text; 2 GPUs "Tesla T4", ~15 GiB free each |
| 2 Code | clones the Lab 7 repository and runs the CPU test | `5/5 checks passed` |
| 3 Data | downloads the 3 MedQA parquet files and checks their SHA-256 | three `OK` lines |
| 4 Smoke test | loads Qwen2.5-7B-Instruct in FP16 with `transformers` on both GPUs, answers 5 debug items (1 greedy + 2 sampled each) | per item: raw text, parsed letter (A–D or None), key, tokens, seconds |

vLLM is **not** installed yet: its version is chosen from cell 1's output (Prompt 08, next step).

In [ ]:
# Cell 1 - diagnostics (no installs)
import importlib, subprocess, sys
def run(cmd):
    print("$", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)
print("python", sys.version)
run("nvidia-smi")
run("df -h / /kaggle/working /tmp | cat")
for p in ["torch", "transformers", "accelerate", "vllm", "huggingface_hub", "numpy", "pandas", "pyarrow", "scipy", "yaml"]:
    try:
        m = importlib.import_module(p); print(f"{p:16s}", getattr(m, "__version__", "?"))
    except Exception as e:
        print(f"{p:16s}", "NOT INSTALLED", type(e).__name__)
import torch
print("cuda available:", torch.cuda.is_available(), "| torch built for CUDA", torch.version.cuda, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(i, torch.cuda.get_device_name(i), "compute capability", torch.cuda.get_device_capability(i),
          f"free {free/2**30:.1f} / {total/2**30:.1f} GiB", "| bf16 supported:", torch.cuda.is_bf16_supported())
from huggingface_hub import model_info
mi = model_info("Qwen/Qwen2.5-7B-Instruct")
print("Qwen2.5-7B-Instruct commit:", mi.sha, "| gated:", getattr(mi, "gated", None))
run("pip index versions vllm 2>/dev/null | head -2")

In [ ]:
# Cell 2 - get the code
REPO_URL = "https://github.com/vibhum-phd/lab7-oncology-llm-voting.git"
import subprocess
print(subprocess.run(f"rm -rf /kaggle/working/repo && git clone --depth 1 {REPO_URL} /kaggle/working/repo && "
                     "git -C /kaggle/working/repo log -1 --format='commit %H %cd'", shell=True, capture_output=True, text=True))
print(subprocess.run("cd /kaggle/working/repo && python tests/test_cpu_env.py", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Cell 3 - data: download MedQA parquet files and verify checksums
import hashlib, os
from huggingface_hub import hf_hub_download
SHA = {"train": "616c13662bb281c57c299c998759e6206bba47365cd37ecb4d52e66712ec34ec",
       "validation": "841a5ffcddd415c9e28c0fd7f412e9ce4ece51c0e705aed7b0bfa324c916be95",
       "test": "debf8e08bce918806287597194c585648dd2b040fea34405f04a72a0dc6e270e"}
os.makedirs("/kaggle/working/repo/data/raw/medqa_4opt_hf", exist_ok=True)
paths = {}
for split, want in SHA.items():
    p = hf_hub_download("GBaker/MedQA-USMLE-4-options-hf", f"default/{split}/0000.parquet",
                        repo_type="dataset", revision="refs/convert/parquet")
    got = hashlib.sha256(open(p, "rb").read()).hexdigest()
    print(split, "OK" if got == want else f"MISMATCH {got}")
    paths[split] = p

In [ ]:
# Cell 4 - smoke test: load the model (FP16, both GPUs) and answer 5 debug items
import json, re, time
import pandas as pd, torch, yaml
from transformers import AutoModelForCausalLM, AutoTokenizer
cfg = yaml.safe_load(open("/kaggle/working/repo/configs/protocol_v1.yaml"))
items = pd.concat([pd.read_parquet(p) for p in paths.values()]).set_index("id")
debug_ids = open("/kaggle/working/repo/data/splits/debug_ids.txt").read().split()[:5]
name, rev = cfg["model"]["name"], mi.sha            # pin to the commit printed in cell 1
t0 = time.time()
tok = AutoTokenizer.from_pretrained(name, revision=rev)
model = AutoModelForCausalLM.from_pretrained(name, revision=rev, torch_dtype=torch.float16, device_map="auto")
print(f"model loaded in {time.time()-t0:.0f}s; device map: {set(model.hf_device_map.values())}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} memory allocated {torch.cuda.memory_allocated(i)/2**30:.1f} GiB")
P, F = re.compile(cfg["parsing"]["primary_regex"]), re.compile(cfg["parsing"]["fallback_regex"])
parse = lambda t: ((P.findall(t) or F.findall(t)) or [None])[-1]
d = cfg["decoding"]
out = open("/kaggle/working/smoke_test_transformers.jsonl", "w")
for iid in debug_ids:
    r = items.loc[iid]
    user = cfg["prompt"]["user_template"].format(question=r.sent1, option_a=r.ending0, option_b=r.ending1,
                                                 option_c=r.ending2, option_d=r.ending3)
    msgs = [{"role": "system", "content": cfg["prompt"]["system"]}, {"role": "user", "content": user}]
    prompt_text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt_text, return_tensors="pt").to(model.device)
    ids = enc["input_ids"]
    for k, (run_name, kw) in enumerate([("greedy", dict(do_sample=False)),
                         ("sample1", dict(do_sample=True, temperature=d["sampled"]["temperature"], top_p=d["sampled"]["top_p"], top_k=d["sampled"]["top_k"])),
                         ("sample2", dict(do_sample=True, temperature=d["sampled"]["temperature"], top_p=d["sampled"]["top_p"], top_k=d["sampled"]["top_k"]))]):
        torch.manual_seed(20261102 + k)        # a different seed for each run
        t = time.time()
        g = model.generate(**enc, max_new_tokens=d["max_tokens"], repetition_penalty=d["repetition_penalty"],
                           pad_token_id=tok.eos_token_id, **kw)
        text = tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True)
        n_new = int(g.shape[1] - ids.shape[1])
        rec = {"id": iid, "run": run_name, "parsed": parse(text), "key": "ABCD"[int(r.label)],
               "new_tokens": n_new, "truncated": n_new >= d["max_tokens"], "seconds": round(time.time() - t, 1)}
        out.write(json.dumps({**rec, "text": text}) + "\n")
        print(rec); print("   ...", text[-300:].replace("\n", " "))
out.close()
print("pip freeze saved:", subprocess.run("pip freeze > /kaggle/working/pip_freeze.txt && wc -l /kaggle/working/pip_freeze.txt", shell=True, capture_output=True, text=True).stdout)

## If something fails (send Claude the full error text)
- **CUDA out of memory** in cell 4: check cell 1 shows 2 GPUs with ~15 GiB free each; restart the session; if it persists, switch to `Qwen/Qwen2.5-7B-Instruct-AWQ` (a logged protocol change).
- **`git clone` fails**: the repository name in cell 2 is wrong, the repository is private, or Internet is off.
- **Download errors / timeouts** in cells 3–4: Internet is off, or Hugging Face is down; rerun the cell.
- **Gated model / 401**: should not happen (the model is not gated); report it.
- **Empty, repeated or garbled text** in cell 4: possible FP16 overflow on T4; report the raw text.
- **Library version clashes** (`ImportError`, `undefined symbol`): copy the full traceback and cell 1 output.